# L04 · Attention 变体

In [ ]:
import sys; sys.path.insert(0, '../src')
import torch
from mha import MHA, kv_cache_size
from mqa import MQA
from gqa import GQA
from mla import MLA, kv_cache_size_mla

x = torch.randn(2, 16, 128)
for cls, kw in [(MHA,{'d_model':128,'n_head':8}), (MQA,{'d_model':128,'n_head':8}),
                 (GQA,{'d_model':128,'n_head':8,'n_kv_head':2})]:
    m = cls(**kw); y = m(x); n_p = sum(p.numel() for p in m.parameters())
    print(f'{cls.__name__:6} out {tuple(y.shape)}  params {n_p}')
mla = MLA(d_model=128, n_head=8, d_low=32); y, c = mla(x)
print(f'MLA    out {tuple(y.shape)}  params {sum(p.numel() for p in mla.parameters())}  c {tuple(c.shape)}')

## KV cache 显存对照（70B 模型，32k context）

In [ ]:
GB = 1024**3
n_layer=80; d_head=128; t=32768
mha = kv_cache_size(n_layer, 64, d_head, t) / GB
gqa = kv_cache_size(n_layer,  8, d_head, t) / GB
mqa = kv_cache_size(n_layer,  1, d_head, t) / GB
mla = kv_cache_size_mla(n_layer, t, 512) / GB
print(f'MHA h=64: {mha:.1f} GB')
print(f'GQA g=8 : {gqa:.1f} GB')
print(f'MQA g=1 : {mqa:.1f} GB')
print(f'MLA d=512: {mla:.1f} GB')